# 07 - Mapa interactiu - Cisjordania

**El producte principal del projecte.** Cartografia interactiva de l'expansio d'assentaments
i outposts a Cisjordania, inspirada en l'enfocament narratiu del visual explainer d'ICG,
construida integrament amb dades propies del pipeline (notebooks 01-06).

**Fonts** (a `data/clean/` i `Datasets/raw/gis/`):
- `settlements_points.csv` - 147 assentaments geolocalitzats
- `outposts_points.csv` - 383 outposts geolocalitzats
- `population_final.csv` - poblacio per assentament i any (per a la mida dels marcadors)
- `demolitions_wb.csv`, `fatalities_settlers_wb.csv` - usats nomes per al panell de xifres clau (veure limitacio mes avall)
- `Settlements_Buildup_PeaceNow.shp` - poligons d'area urbanitzada per assentament (Peace Now)
- `Barrier_Jan2018.shp` - traçat de la barrera (Peace Now, gener 2018), capa de referencia estatica

**Output:**
- `07_map.html` - mapa interactiu autonom, per obrir directament al navegador

**Nota d'estil:** l'esquema de colors i l'ordre de capes (base -> poligons -> barrera de
referencia -> punts) s'inspira en el visual explainer d'OCHA (`Factsheet_Booklet_Movement_and_Access.pdf`),
nomes com a referencia de presentacio - no se n'extreu cap dada ni s'hi reprodueix cap xifra.

---
## Disponible en aquesta versio (versio final)
- Capa Assentaments: mida proporcional a la poblacio (2024), color per tipologia, popup net
- Capa Outposts: simbol diferenciat (diamant) dels assentaments, color per tipologia, popup net
- Capa Area urbanitzada dels assentaments (poligons Peace Now), amb superficie al popup
- Capa Barrera (Peace Now, gener 2018) - nomes de referencia geografica, no s'actualitza
- Totes les capes activables/desactivables per separat (LayerControl)
- Panell flotant de xifres clau + llegenda
- Pantalla completa per a la defensa del projecte
- Export HTML autonom

## Pendent - documentat, no simulat (fora d'abast d'aquesta versio final)
- **Demolicions i victimes com a capa geografica propia:** `demolitions_wb.csv` (431 localitats)
  i `fatalities_settlers_wb.csv` identifiquen les localitats palestines pel seu nom, mentre
  que la geodada disponible nomes conte noms de consells regionals israelians
  (Shomron, Gush Etzion...). No hi ha cap camp comu per fer el join - cal un gasetter
  de localitats palestines amb coordenades.
- **Green Line, Arees A/B/C, checkpoints:** cap fitxer GIS disponible localment. No s'afegeixen
  en aquesta versio (instruccio explicita: no ampliar mes enlla del que ja tenim).


## 1. Importacio de llibreries

In [33]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
from folium.plugins import Fullscreen
import os

print("Llibreries carregades correctament")

Llibreries carregades correctament


## 2. Configuracio de paths

In [34]:
CLEAN = "data/clean"
OUT_DIR = "outputs"

FILE_SETTLEMENTS = f"{CLEAN}/settlements_points.csv"
FILE_OUTPOSTS    = f"{CLEAN}/outposts_points.csv"
FILE_POPULATION  = f"{CLEAN}/population_final.csv"
FILE_DEMOLITIONS = f"{CLEAN}/demolitions_wb.csv"
FILE_FATALITIES  = f"{CLEAN}/fatalities_settlers_wb.csv"

OUT_HTML = f"{OUT_DIR}/07_map.html"

# Capes GIS (shapefiles Peace Now), fora de data/clean perque son fitxers geometrics originals
RAW_GIS = "../Datasets/raw/gis"
FILE_BUILDUP = f"{RAW_GIS}/Settlements_Buildup_PeaceNow.shp"
FILE_BARRIER = f"{RAW_GIS}/Barrier_Jan2018.shp"

os.makedirs(OUT_DIR, exist_ok=True)
print("Paths configurats:", os.path.abspath(OUT_DIR))

Paths configurats: c:\Users\a-iba\OneDrive\Documentos\Sprint13\Notebook\outputs


---
## Seccio A - Preparacio: Assentaments + poblacio

**Nota metodologica important:** per acolorir la mida dels marcadors per poblacio, faig
join de `settlements_points.csv` amb `population_final.csv` de l'any **2024**, no 2025.
2024 es l'ultim any 100% Peace Now (la mateixa font que `settlements_points.csv`), cosa que
dona un 86% de coincidencia de noms (126/147). Amb 2025 (font JVL) la coincidencia cauria al
50% per diferencies de transliteracio entre fonts. Els assentaments sense coincidencia es
mostren igualment (mida minima fixa), mai s'exclouen ni s'inventa una xifra.


In [35]:
df_settlements = pd.read_csv(FILE_SETTLEMENTS)
df_pop = pd.read_csv(FILE_POPULATION)

pop_2024 = df_pop[df_pop["year"] == 2024][["settlement", "population"]]

df_settlements = df_settlements.merge(pop_2024, left_on="name", right_on="settlement", how="left")
df_settlements = df_settlements.drop(columns=["settlement"])

n_matched = df_settlements["population"].notna().sum()
print(f"Assentaments amb poblacio 2024 assignada: {n_matched}/{len(df_settlements)}")
print(f"Sense coincidencia (mida minima): {df_settlements[df_settlements['population'].isna()]['name'].tolist()}")

Assentaments amb poblacio 2024 assignada: 126/147
Sense coincidencia (mida minima): ['Alon', 'Elmatan', 'Beit Al-Baraka', "Giv'on", "Gva'ot", 'Har Shmuel', 'Hebron', 'Horesh Yaron', 'Tal Menashe', 'New Migron', "Ma'ale Shomron", 'Mitzpe Eshtamoa', 'Nofei Prat', 'Nahalei Tal (Kerem Reim)', 'Nirit', 'Nerya', 'Ofarim', 'Etz Efraim', 'Shvut Rachel', 'Shani', "Sha'arei Tikva"]


In [36]:
# Escala de radi: arrel quadrada de la poblacio (evita que els grans dominin visualment)
POP_MIN_RADIUS = 4
POP_MAX_RADIUS = 22

pop_valid = df_settlements["population"].dropna()
sqrt_pop = np.sqrt(df_settlements["population"])
sqrt_min, sqrt_max = np.sqrt(pop_valid.min()), np.sqrt(pop_valid.max())

def scale_radius(sqrt_val):
    if pd.isna(sqrt_val):
        return POP_MIN_RADIUS
    return POP_MIN_RADIUS + (sqrt_val - sqrt_min) / (sqrt_max - sqrt_min) * (POP_MAX_RADIUS - POP_MIN_RADIUS)

df_settlements["marker_radius"] = sqrt_pop.apply(scale_radius)

# Color per tipologia (urban_pattern -> category)
SETTLEMENT_COLORS = {
    "Community":         "#4C72B0",
    "Urban":             "#C44E52",
    "Moshav":            "#55A868",
    "Kibbutz":           "#8172B3",
    "Cooperative":       "#CCB974",
    "Moshav/Community":  "#64B5CD",
}
df_settlements["marker_color"] = df_settlements["category"].map(SETTLEMENT_COLORS).fillna("#999999")

print(df_settlements[["name", "population", "marker_radius", "category"]].head())

           name  population  marker_radius   category
0  Avnei Hefetz      2536.0       6.480519      Urban
1         Ovnat       286.0       4.355628  Community
2         Adora       541.0       4.758907  Community
3        Oranit      9397.0       9.439697      Urban
4        Itamar      1596.0       5.819254  Community


---
## Seccio B - Preparacio: Outposts


In [37]:
df_outposts = pd.read_csv(FILE_OUTPOSTS)

OUTPOST_COLORS = {
    "Outpost":                "#DD8452",
    "Farm Outpost":            "#937860",
    "Farm Outpost - Yeshiva":  "#DA8BC3",
}
df_outposts["marker_color"] = df_outposts["category"].map(OUTPOST_COLORS).fillna("#999999")

print(f"Outposts: {len(df_outposts)}")
print(df_outposts["category"].value_counts())

Outposts: 383
category
Farm Outpost              222
Outpost                   160
Farm Outpost - Yeshiva      1
Name: count, dtype: int64


---
## Seccio C - Xifres clau (per al panell flotant)

Aquestes xifres **no es geolocalitzen** (per aixo no formen una capa propia al mapa), pero
aporten context immediat sense necessitat del gasetter pendent. Es l'afegit que et comentava:
dona al mapa una lectura completa del fenomen encara que dues de les cinc dimensions
(demolicions, victimes) no es puguin representar espacialment en aquesta versio.


In [38]:
df_dem = pd.read_csv(FILE_DEMOLITIONS)
df_fat = pd.read_csv(FILE_FATALITIES)

stats = {
    "pop_total_2024": int(pop_2024["population"].sum()),
    "n_settlements":  len(df_settlements),
    "n_outposts":     len(df_outposts),
    "n_demolitions":  len(df_dem),
    "housing_units":  int(df_dem["housing_units"].sum()),
    "n_fatalities":   len(df_fat),
}
stats

{'pop_total_2024': 496852,
 'n_settlements': 147,
 'n_outposts': 383,
 'n_demolitions': 5465,
 'housing_units': 8206,
 'n_fatalities': 118}

*(Les xifres d'area urbanitzada i longitud de barrera es calculen mes endavant, un cop
carregades les capes GIS a la Seccio D2/D3, i s'afegeixen al mateix diccionari `stats`.)*

---
## Seccio D - Construccio del mapa base


In [39]:
CENTER_LAT, CENTER_LON = 31.95, 35.23

m = folium.Map(
    location=[CENTER_LAT, CENTER_LON],
    zoom_start=9,
    tiles="CartoDB positron",
    control_scale=True,
)

folium.TileLayer("OpenStreetMap", name="OpenStreetMap (base alternativa)").add_to(m)

Fullscreen(position="topleft", title="Pantalla completa", title_cancel="Sortir de pantalla completa").add_to(m)

print("Mapa base creat")

Mapa base creat


---
## Seccio D2 - Capa: Area urbanitzada dels assentaments (poligons)

Poligons de Peace Now amb l'area edificada real de cada assentament (no un punt, sino la
seva petjada sobre el territori). El shapefile ve en coordenades UTM (EPSG:32636); es
reprojecta a WGS84 (EPSG:4326) per poder-lo dibuixar amb folium/Leaflet.

Aquesta capa s'afegeix al mapa **abans** que els punts (Seccio E) perque, en folium, les
capes afegides mes tard es dibuixen per sobre. Aixi els poligons queden de fons i els
marcadors de punts sempre son clicables per damunt seu.


In [40]:
gdf_buildup = gpd.read_file(FILE_BUILDUP)
print(f"CRS original: {gdf_buildup.crs}")
print(f"Poligons: {len(gdf_buildup)}")

gdf_buildup = gdf_buildup.to_crs(4326)

# Netejar noms (alguns porten salts de linia de la font) i marcar els que no en tenen
gdf_buildup["Name"] = gdf_buildup["Name"].astype(str).str.strip().replace({"None": None, "nan": None})
gdf_buildup["area_ha"] = gdf_buildup["Shape_Area"] / 10_000  # m2 -> hectarees

print(f"Sense nom: {gdf_buildup['Name'].isna().sum()} / {len(gdf_buildup)}")
print(f"Area total urbanitzada: {gdf_buildup['area_ha'].sum():,.0f} ha")

CRS original: EPSG:32636
Poligons: 201
Sense nom: 24 / 201
Area total urbanitzada: 7,094 ha


In [41]:
BUILDUP_FILL   = "#8B2E2E"   # maroon, inspirat en la simbologia d'assentaments d'OCHA
BUILDUP_BORDER = "#5C1F1F"

fg_buildup = folium.FeatureGroup(name="Area urbanitzada (assentaments)", show=True)

for _, row in gdf_buildup.iterrows():
    name_txt = row["Name"] if row["Name"] else "Sense nom (Peace Now)"
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; min-width: 170px;">
        <b>{name_txt}</b><br>
        <span style="color:#666;">Area urbanitzada</span>
        <hr style="margin: 4px 0;">
        Superficie: <b>{row['area_ha']:.1f} ha</b>
    </div>
    """
    folium.GeoJson(
        row["geometry"],
        style_function=lambda x: {
            "fillColor": BUILDUP_FILL, "color": BUILDUP_BORDER,
            "weight": 1, "fillOpacity": 0.55,
        },
        popup=folium.Popup(popup_html, max_width=220),
    ).add_to(fg_buildup)

fg_buildup.add_to(m)
print(f"Capa Area urbanitzada afegida: {len(gdf_buildup)} poligons")

Capa Area urbanitzada afegida: 201 poligons


---
## Seccio D3 - Capa: Barrera (Peace Now, gener 2018)

Capa de referencia geografica estatica, tal com demanaves: **no s'actualitza**, nomes
s'afegeix perque doni context territorial. Es distingeix per `Status` (Constructed / Under
Construction / Projected) amb un estil de linia diferent per a cadascun, seguint la
convencio visual habitual d'OCHA (linia continua = construida, discontinua = en curs o
projectada).


In [42]:
gdf_barrier = gpd.read_file(FILE_BARRIER)
gdf_barrier = gdf_barrier.to_crs(4326)

print(f"Trams de barrera: {len(gdf_barrier)}")
print(gdf_barrier["Status"].value_counts())

# Longitud aproximada (projeccio UTM original, abans de reprojectar, dona metres reals)
length_km = gpd.read_file(FILE_BARRIER)["Shape_Leng"].sum() / 1000
print(f"Longitud total (segons el fitxer, gener 2018): {length_km:,.0f} km")

Trams de barrera: 261
Status
Constructed           181
Projected              52
Under Construction     28
Name: count, dtype: int64
Longitud total (segons el fitxer, gener 2018): 712 km


In [43]:
BARRIER_STYLE = {
    "Constructed":         {"color": "#1A1A1A", "weight": 2.5, "dashArray": None},
    "Under Construction":  {"color": "#1A1A1A", "weight": 2,   "dashArray": "6,4"},
    "Projected":           {"color": "#1A1A1A", "weight": 2,   "dashArray": "2,4"},
}
DEFAULT_BARRIER_STYLE = {"color": "#1A1A1A", "weight": 2, "dashArray": "2,4"}

fg_barrier = folium.FeatureGroup(name="Barrera (referencia, gener 2018)", show=False)

for _, row in gdf_barrier.iterrows():
    style = BARRIER_STYLE.get(row["Status"], DEFAULT_BARRIER_STYLE)
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px;">
        <b>Barrera</b><br>
        Estat: {row['Status']}<br>
        Tipus: {row['Type'] if pd.notna(row['Type']) else 'n/d'}
    </div>
    """
    folium.GeoJson(
        row["geometry"],
        style_function=lambda x, style=style: style,
        popup=folium.Popup(popup_html, max_width=200),
    ).add_to(fg_barrier)

fg_barrier.add_to(m)
print(f"Capa Barrera afegida: {len(gdf_barrier)} trams (per defecte desactivada, es referencia)")

Capa Barrera afegida: 261 trams (per defecte desactivada, es referencia)


---
## Seccio E - Capes: Assentaments i Outposts

Popups nets amb la informacio rellevant nomes (no totes les columnes disponibles), tal com
demanaves: llegibilitat per davant de la quantitat d'informacio.


In [44]:
fg_settlements = folium.FeatureGroup(name="Assentaments", show=True)

for _, row in df_settlements.iterrows():
    pop_txt = f"{int(row['population']):,}" if pd.notna(row["population"]) else "no disponible"
    year_txt = int(row["year_established"]) if pd.notna(row["year_established"]) else "desconegut"
    dist_txt = f"{row['dist_green_line_km']:.1f} km" if pd.notna(row["dist_green_line_km"]) else "n/d"

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; min-width: 190px;">
        <b style="font-size: 14px;">{row['name']}</b><br>
        <span style="color:#666;">Assentament ({row['category'] if pd.notna(row['category']) else 'sense categoritzar'})</span>
        <hr style="margin: 4px 0;">
        Poblacio (2024): <b>{pop_txt}</b><br>
        Any de fundacio: {year_txt}<br>
        Consell regional: {row['municipality']}<br>
        Dist. Green Line: {dist_txt}
    </div>
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=row["marker_radius"],
        color="#333333",
        weight=0.6,
        fill=True,
        fill_color=row["marker_color"],
        fill_opacity=0.85,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=row["name"],
    ).add_to(fg_settlements)

fg_settlements.add_to(m)
print(f"Capa Assentaments afegida: {len(df_settlements)} punts")

Capa Assentaments afegida: 147 punts


In [45]:
fg_outposts = folium.FeatureGroup(name="Outposts", show=True)

DIAMOND_SIZE = 10

for _, row in df_outposts.iterrows():
    year_txt = int(row["year_established"]) if pd.notna(row["year_established"]) else "desconegut"
    nearest_txt = row["nearest_settlement"] if pd.notna(row["nearest_settlement"]) else "n/d"
    district_txt = row["district"] if pd.notna(row["district"]) else "n/d"

    popup_html = f"""
    <div style="font-family: Arial, sans-serif; font-size: 13px; min-width: 190px;">
        <b style="font-size: 14px;">{row['name']}</b><br>
        <span style="color:#666;">Outpost ({row['category']})</span>
        <hr style="margin: 4px 0;">
        Any de fundacio: {year_txt}<br>
        Assentament mes proper: {nearest_txt}<br>
        Districte: {district_txt}
    </div>
    """

    icon_html = f"""
    <div style="
        width: {DIAMOND_SIZE}px; height: {DIAMOND_SIZE}px;
        background-color: {row['marker_color']};
        border: 1px solid #333333;
        transform: rotate(45deg);
    "></div>
    """

    folium.Marker(
        location=[row["lat"], row["lon"]],
        icon=folium.DivIcon(html=icon_html, icon_size=(DIAMOND_SIZE, DIAMOND_SIZE), icon_anchor=(DIAMOND_SIZE // 2, DIAMOND_SIZE // 2)),
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=row["name"],
    ).add_to(fg_outposts)

fg_outposts.add_to(m)
print(f"Capa Outposts afegida: {len(df_outposts)} punts (simbol diamant, diferenciat dels cercles d'assentaments)")

Capa Outposts afegida: 383 punts (simbol diamant, diferenciat dels cercles d'assentaments)


---
## Seccio F - Capes pendents (fora d'abast d'aquesta versio)

Per instruccio explicita, aquesta versio final **no** incorpora Green Line, Arees A/B/C ni
checkpoints perque no hi ha cap fitxer GIS local per a aquestes capes - i no se n'ha
d'anar a buscar cap. Es deixen nomes les funcions ja escrites (no executades) perque, el
dia que hi hagi els fitxers, es puguin cridar sense tocar la resta del notebook.


In [46]:
def add_line_layer(m, geojson_path, name, color="#000000", weight=2, dash_array=None):
    """Per a Green Line (LineString/MultiLineString GeoJSON)."""
    import json
    with open(geojson_path) as f:
        geo = json.load(f)
    style = {"color": color, "weight": weight}
    if dash_array:
        style["dashArray"] = dash_array
    fg = folium.FeatureGroup(name=name, show=False)
    folium.GeoJson(geo, style_function=lambda x, style=style: style).add_to(fg)
    fg.add_to(m)
    return fg


def add_polygon_layer(m, geojson_path, name, color_property=None, color_map=None, default_color="#888888", fill_opacity=0.25):
    """Per a Arees A/B/C (Polygon/MultiPolygon GeoJSON, un color per tipus d'area)."""
    import json
    with open(geojson_path) as f:
        geo = json.load(f)

    def style_function(feature):
        val = feature["properties"].get(color_property) if color_property else None
        color = (color_map or {}).get(val, default_color)
        return {"fillColor": color, "color": color, "weight": 1, "fillOpacity": fill_opacity}

    fg = folium.FeatureGroup(name=name, show=False)
    folium.GeoJson(geo, style_function=style_function).add_to(fg)
    fg.add_to(m)
    return fg


def add_point_layer_from_geojson(m, geojson_path, name, icon_color="black", icon="info-sign"):
    """Per a checkpoints (Point GeoJSON)."""
    import json
    with open(geojson_path) as f:
        geo = json.load(f)

    fg = folium.FeatureGroup(name=name, show=False)
    for feature in geo["features"]:
        lon, lat = feature["geometry"]["coordinates"]
        props = feature.get("properties", {})
        folium.Marker(
            location=[lat, lon],
            popup=str(props),
            icon=folium.Icon(color=icon_color, icon=icon),
        ).add_to(fg)
    fg.add_to(m)
    return fg


# Exemples d'us (descomentar nomes quan hi hagi els fitxers a RAW_GIS - no abans):
# add_line_layer(m, f"{RAW_GIS}/green_line.geojson", "Green Line", color="#000000", dash_array="6,4")
# add_polygon_layer(m, f"{RAW_GIS}/areas_abc.geojson", "Arees A/B/C",
#                    color_property="AREA", color_map={"A": "#2ca02c", "B": "#ff7f0e", "C": "#d62728"})
# add_point_layer_from_geojson(m, f"{RAW_GIS}/checkpoints.geojson", "Checkpoints", icon_color="red", icon="road")

print("Funcions de capes GIS pendents definides (no executades).")

Funcions de capes GIS pendents definides (no executades).


In [47]:
def add_geocoded_events_layer(m, df, name, lat_col="lat", lon_col="lon", color="#C44E52", radius=4):
    """
    Per a demolicions/victimes un cop es disposi d'un gasetter de localitats palestines
    amb coordenades i s'hagi fet el join per nom de localitat (veure nota Seccio F).
    """
    fg = folium.FeatureGroup(name=name, show=False)
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=radius,
            color=color,
            fill=True,
            fill_opacity=0.7,
        ).add_to(fg)
    fg.add_to(m)
    return fg

# Exemple d'us un cop demolitions_wb.csv / fatalities_settlers_wb.csv tinguin lat/lon:
# add_geocoded_events_layer(m, df_dem_geocoded, "Demolicions", color="#C44E52")
# add_geocoded_events_layer(m, df_fat_geocoded, "Victimes", color="#55A868")

print("Funcio per a capes de violencia geolocalitzada definida (no executada).")

Funcio per a capes de violencia geolocalitzada definida (no executada).


In [48]:
# Ampliar les xifres clau amb les noves capes GIS
stats["buildup_area_ha"] = round(gdf_buildup["area_ha"].sum())
stats["barrier_km"] = round(length_km)
stats

{'pop_total_2024': 496852,
 'n_settlements': 147,
 'n_outposts': 383,
 'n_demolitions': 5465,
 'housing_units': 8206,
 'n_fatalities': 118,
 'buildup_area_ha': 7094,
 'barrier_km': 712}

---
## Seccio G - Panell de xifres clau i llegenda

Dues capses HTML flotants: xifres clau (superior dreta) i llegenda de simbols (inferior
esquerra). Es l'element que dona al mapa aspecte de producte acabat en lloc de "notebook amb
un mapa a sobre".


In [49]:
stats_html = f"""
<div style="
    position: fixed; top: 12px; right: 12px; z-index: 9999;
    background-color: white; padding: 12px 16px; border-radius: 6px;
    box-shadow: 0 1px 6px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif; font-size: 12.5px; line-height: 1.5;
    max-width: 210px;
">
    <b style="font-size: 14px;">Cisjordania - xifres clau</b><br>
    <span style="color:#666; font-size: 11px;">Assentaments i outposts</span>
    <hr style="margin: 6px 0;">
    Poblacio colons (2024): <b>{stats['pop_total_2024']:,}</b><br>
    Assentaments: <b>{stats['n_settlements']}</b><br>
    Outposts: <b>{stats['n_outposts']}</b><br>
    Area urbanitzada: <b>{stats['buildup_area_ha']:,} ha</b><br>
    Barrera (referencia 2018): <b>{stats['barrier_km']:,} km</b>
    <hr style="margin: 6px 0;">
    <span style="color:#666; font-size: 11px;">Impacte documentat (no geolocalitzat, veure nota)</span><br>
    Demolicions (2006-2026): <b>{stats['n_demolitions']:,}</b><br>
    Habitatges demolits: <b>{stats['housing_units']:,}</b><br>
    Victimes mortals (2000-2026): <b>{stats['n_fatalities']:,}</b>
</div>
"""

m.get_root().html.add_child(folium.Element(stats_html))
print("Panell de xifres clau afegit")

Panell de xifres clau afegit


In [50]:
legend_html = """
<div style="
    position: fixed; bottom: 24px; left: 12px; z-index: 9999;
    background-color: white; padding: 10px 14px; border-radius: 6px;
    box-shadow: 0 1px 6px rgba(0,0,0,0.3);
    font-family: Arial, sans-serif; font-size: 12px; line-height: 1.6;
">
    <b>Llegenda</b><br>
    <span style="display:inline-block; width:12px; height:12px; border-radius:50%; background:#4C72B0; margin-right:6px;"></span>Assentament (mida = poblacio)<br>
    <span style="display:inline-block; width:10px; height:10px; background:#DD8452; transform:rotate(45deg); margin-right:8px;"></span>Outpost<br>
    <span style="display:inline-block; width:14px; height:10px; background:#8B2E2E; opacity:0.6; margin-right:6px; vertical-align:middle;"></span>Area urbanitzada<br>
    <span style="display:inline-block; width:14px; height:0px; border-top:2px dashed #1A1A1A; margin-right:6px; vertical-align:middle;"></span>Barrera (referencia)<br>
    <span style="color:#888; font-size: 11px;">Color de punts = tipologia (veure popup)</span>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))
print("Llegenda afegida")

Llegenda afegida


---
## Seccio H - Capes i export


In [51]:
folium.LayerControl(collapsed=False).add_to(m)

# Ajustar la vista a l'extensio real de totes les capes (punts + poligons + barrera)
all_lats = pd.concat([df_settlements["lat"], df_outposts["lat"]])
all_lons = pd.concat([df_settlements["lon"], df_outposts["lon"]])
buildup_bounds = gdf_buildup.total_bounds   # [minx, miny, maxx, maxy] = [lon_min, lat_min, lon_max, lat_max]
barrier_bounds = gdf_barrier.total_bounds

lat_min = min(all_lats.min(), buildup_bounds[1], barrier_bounds[1])
lat_max = max(all_lats.max(), buildup_bounds[3], barrier_bounds[3])
lon_min = min(all_lons.min(), buildup_bounds[0], barrier_bounds[0])
lon_max = max(all_lons.max(), buildup_bounds[2], barrier_bounds[2])

m.fit_bounds([[lat_min, lon_min], [lat_max, lon_max]])

m.save(OUT_HTML)
print(f"Mapa exportat: {OUT_HTML}")

Mapa exportat: outputs/07_map.html


In [52]:
m

---
## Resum

| Element | Estat |
|---|---|
| Capa Assentaments (mida=poblacio, color=tipologia) | Completa |
| Capa Outposts (simbol diamant, color=tipologia) | Completa |
| Capa Area urbanitzada (poligons Peace Now) | Completa |
| Capa Barrera (Peace Now, gener 2018, referencia estatica) | Completa |
| Panell de xifres clau + llegenda | Completa (actualitzat amb area i barrera) |
| Pantalla completa | Completa |
| Green Line / Arees A/B/C / Checkpoints | Fora d'abast d'aquesta versio (sense fitxers locals) |
| Demolicions / Victimes com a capa geografica | Pendent d'un gasetter de localitats palestines |

**Aquesta es la versio final del mapa del projecte.** Cap component queda simulat o
inventat: tot el que es mostra prove directament dels fitxers de dades disponibles.

**Fitxer generat:** `outputs/07_map.html` - autonom, es pot obrir directament al navegador
o incrustar a la presentacio final.
